In [1]:
%%bash
mkdir -p frames
if [ -z "$( ls -A 'frames' )" ]; then
    ffmpeg -i original.mp4 -vf fps=24 frames/frame%04d.png
else
    echo Frames folder not empty. Skipping ffmpeg conversion
fi

Frames folder not empty. Skipping ffmpeg conversion


In [2]:
HEIGHT = 48
WIDTH = 36
FRAMES = 5258 # 5258
FPS = 1 # 24
MODE = 2 # 1 for sequence per frame, 2 for sequence per coordinate

In [ ]:
!mkdir -p queries
!touch queries/.gitkeep
!rm queries/query_*.fa
from query_generator import generate_queries_by_frame, generate_queries_by_coordinate
if MODE == 1:
    generate_queries_by_frame(HEIGHT, WIDTH, FRAMES, FPS)
elif MODE == 2:
    generate_queries_by_coordinate(HEIGHT, WIDTH, FRAMES, FPS)

In [ ]:
%%writefile custom.matrix
   A  C  G  T
A  3 -6  3 -6
C -6  3 -6  3
G  3 -6  3 -6
T -6  3 -6  3

In [ ]:
def get_sequence(data):
    if data["status"] != "Sequence found":
        return ""
    s = data["sequence"]
    if data["rev-comp"]:
        return s[::-1].replace("A", "_").replace("T", "A").replace("_", "T").replace("C", "_").replace("G", "C").replace("_", "G")
    return s

sequences = {}

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

plt.rcParams['font.family'] = 'STIXGeneral'

fig, ax = plt.subplots()
ax.axis("off")

fig_text = {}
for x in range(WIDTH):
    for y in range(HEIGHT):
        fig_text[(x, y)] = fig.text(x/WIDTH, 1-y/HEIGHT, "N")
        
plt.tight_layout()
        
def clear():
    for tx in fig_text.values():
        tx.set_text("-")
        tx.set_color((0,0,0))

def draw(i):
    if MODE == 1:
        if sequences[i] == "":
            return
        print(f"Drawing frame {i} in frame mode", end="\r", flush=True)
        remainder = len(sequences[i]) % WIDTH
        pad_width = (WIDTH - remainder) % WIDTH
        sequence = np.pad(np.array(list(sequences[i])), (0, pad_width), mode='constant', constant_values='-').reshape((-1, WIDTH))
        for x in range(WIDTH):
            for y in range(HEIGHT):
                base = sequence[y,x]
                color = (1,0,0) if base in "CT" else (0,0,1) if base in "AG" else (0,0,0)
                fig_text[(x, y)].set_text(base)
                fig_text[(x, y)].set_color(color)
    elif MODE == 2:
        print(f"Drawing frame {i} in coordinate mode", end="\r", flush=True)
        for x, y in fig_text.keys():
            sequence = sequences[(x, y)]
            if sequence != "":
                base = sequence[i] if i<len(sequence) else '-'
                color = (1,0,0) if base in "CT" else (0,0,1) if base in "AG" else (0,0,0)
                fig_text[(x, y)].set_text(base)
                fig_text[(x, y)].set_color(color)
                

In [ ]:
# Sample
if MODE == 1:
    sample_frame = "3217"
    !glsearch36 -s custom.matrix -b 1 -d 1 queries/query_{sample_frame}.fa chr1.fna > sample.output

In [ ]:
# Sample
if MODE == 1:
    from IPython.display import Image
    Image(filename=f'frames/frame{sample_frame}.png')

In [ ]:
# Sample
if MODE == 1:
    with open(f"queries/query_{sample_frame}.fa", "r") as f:
        f.readline()
        sequences[-2] = f.readline()
    draw(-2)
    sequences.pop(-2)
    fig

In [ ]:
if MODE == 1:
    clear()

In [ ]:
# Sample
if MODE == 1:
    from output_parser import parse
    sequences[-1] = get_sequence(parse("sample.output"))
    draw(-1)
    sequences.pop(-1)
    fig

In [ ]:
if MODE == 1:
    clear()

In [ ]:
%%bash
mkdir -p outputs
touch outputs/.gitkeep
. align.sh

In [ ]:
from output_parser import parse_to_json_files
parse_to_json_files()

In [ ]:
import glob
import json

data = {}

if MODE == 1:
    i = 0
    for filename in glob.glob("outputs/fasta_*.json"):
        with open(filename, "r") as f:
            data[i] = json.loads(f.read())
        i += 1
elif MODE == 2:
    for x in range(WIDTH):
        for y in range(HEIGHT):
            try:
                with open(f"outputs/fasta_{y:02d}_{x:02d}.json", "r") as f:
                    data[(x, y)] = json.loads(f.read())
            except FileNotFoundError:
                    data[(x, y)] = {"status": "File not found"}

print(len(data), "files loaded")

In [ ]:
sequences = {}

if MODE == 1:
    for i in range(len(data)):
        sequences[i] = get_sequence(data[i])
    print(sequences[0])
elif MODE == 2:
    for x in range(WIDTH):
        for y in range(HEIGHT):
            sequences[(x, y)] = get_sequence(data[(x, y)])
    print(sequences[(0, 0)])

In [ ]:
ani = animation.FuncAnimation(fig=fig, func=draw, frames=FRAMES*FPS//24, interval=1)

FFwriter = animation.FFMpegWriter(fps=FPS)
ani.save('animation.mp4', writer=FFwriter)
fig

In [ ]:
from moviepy import VideoFileClip, clips_array

clip1 = VideoFileClip("original.mp4")
clip2 = VideoFileClip("animation.mp4")

final_clip = clips_array([[clip1, clip2]])

final_clip.write_videofile("final.mp4", fps=24)

In [ ]:
from IPython.display import Video
Video("final.mp4")

In [ ]:
del ani
plt.close(fig)